# Homework 7 — Sourcing data

About 60 minutes, plus: **choose your capstone option** (A, B or C) and read its brief in `notebooks/capstone_<option>.ipynb`.

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

In [ ]:
# Helpers from session 7. Just run this cell.
for package in ["boto3"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)

import csv
import fnmatch
import json
import os
import shutil
import sqlite3
from contextlib import closing
from datetime import datetime, timezone
from types import SimpleNamespace

import boto3
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError

WORK = DATA.parent / "work_sourcing"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()
BUCKET = "noaa-gsod-pds"
STATIONS = {"11036099999": "AT", "11816099999": "SK", "12839099999": "HU",
            "11518099999": "CZ", "10385099999": "DE", "12375099999": "PL"}


class LocalSFTP:
    def __init__(self, root):
        self.root = Path(root)

    def listdir_attr(self, path="."):
        folder = self.root / path.strip("/")
        return [SimpleNamespace(filename=p.name, st_size=p.stat().st_size, st_mtime=p.stat().st_mtime) for p in sorted(folder.iterdir())]

    def get(self, remotepath, localpath):
        shutil.copy(self.root / remotepath.strip("/"), localpath)


class LocalS3:
    def __init__(self, root):
        self.root = Path(root)

    def list_objects_v2(self, Bucket, Prefix="", **kwargs):
        base = self.root / Bucket
        keys = sorted(p.relative_to(base).as_posix() for p in base.rglob("*") if p.is_file())
        contents = [{"Key": k, "Size": (base / k).stat().st_size} for k in keys if k.startswith(Prefix)]
        return {"Contents": contents, "KeyCount": len(contents), "IsTruncated": False}

    def download_file(self, Bucket, Key, Filename):
        shutil.copy(self.root / Bucket / Key, Filename)


def connect_s3():
    s3 = boto3.client("s3", region_name="us-east-1",
                      config=Config(signature_version=UNSIGNED, connect_timeout=5, retries={"max_attempts": 1}))
    try:
        s3.list_objects_v2(Bucket=BUCKET, Prefix="2024/11036", MaxKeys=1)
        print("Connected to the real public S3 bucket")
        return s3
    except (BotoCoreError, ClientError) as error:
        print(f"S3 not reachable ({type(error).__name__}) - using the local copy")
        return LocalS3(DATA / "s3_fallback")


partner_sftp = LocalSFTP(DATA / "sftp_drop")
s3 = connect_s3()

### Exercise 1 — A data contract (no code)
Answer the source checklist for the **Danube Records** release feed, as far as you can tell from the files and `README.txt`
(guess sensibly where you can't know, and mark it as an open question for the partner):

1. Format and columns? 2. How often, how much? 3. How do we authenticate? 4. How do we know what is new?
5. What do we do with a late, broken or duplicated delivery? 6. Who do we contact when it breaks?

*Your answers here.*

### Exercise 2 — A download manifest
Extend `pull_new_files` from session 7 so that it also **appends one line per downloaded file** to `local_dir / "_manifest.csv"`
with `file`, `size` and `downloaded_at` (UTC, ISO format). This is lineage for files: you can always prove what you received and when.
Run it twice and print the manifest: it should contain 5 lines plus the header.

In [ ]:
def pull_new_files(sftp, remote_dir, local_dir, pattern):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    # YOUR CODE HERE
    ...


landing = WORK / "danube_records"
print(pull_new_files(partner_sftp, "/danube_records", landing, "releases_*.csv"))
print(pull_new_files(partner_sftp, "/danube_records", landing, "releases_*.csv"))
print((landing / "_manifest.csv").read_text())

### Exercise 3 — Listing a big bucket
`list_objects_v2` returns at most 1,000 keys. When there are more, the response has `IsTruncated = True` and a
`NextContinuationToken`; pass it as `ContinuationToken=...` in the next call.

Write `list_all_keys(s3, bucket, prefix)` that returns **all** keys, following the tokens. Try it with prefix `"2024/11"`
(stations whose id starts with 11: Austria, Switzerland, Czechia, Slovakia, ...). How many keys and how many calls?
(With the local fallback there are only a few files and one call.)

In [ ]:
def list_all_keys(s3, bucket, prefix):
    # YOUR CODE HERE
    ...


keys = list_all_keys(s3, BUCKET, "2024/11")
print(keys[:5])

### Exercise 4 — Weather facts
Download the six station files (as in session 7) into `WORK / "noaa"`, clean them (Fahrenheit → Celsius, sentinel `9999.9` → null),
and answer:
1. The **hottest** and the **coldest** day of 2024 per country (`MAX` and `MIN` columns are the daily max/min temperature in °F).
2. A table with one row per country and one column per month with the average temperature (hint: `pivot`).

In [ ]:
# YOUR CODE HERE

### Exercise 5 — A nightly database pull with saved state
1. Copy the billing database to `WORK / "billing.sqlite"` (we will change it, so never use the original).
2. Write `nightly_pull(db_path, state_file)`: read the watermark from `state_file` (JSON; start with `"1970-01-01 00:00:00"` if it doesn't exist),
   pull the changed rows (read-only connection, query parameter!), save the new watermark (largest `updated_at`) if there were rows,
   and return the number of rows.
3. Run it twice. Then run the given "app activity" cell, which updates 5 subscriptions, and run `nightly_pull` again. You should get exactly 5.

In [ ]:
db_path = WORK / "billing.sqlite"
shutil.copy(DATA / "db" / "wavely_billing.sqlite", db_path)
state_file = WORK / "billing_state.json"


def nightly_pull(db_path, state_file):
    # YOUR CODE HERE
    ...


nightly_pull(db_path, state_file)  # first run: everything (800 rows)
nightly_pull(db_path, state_file)  # nothing changed: 0 rows

In [ ]:
# "App activity": 5 users upgrade to premium. Run this, then run nightly_pull again.
with closing(sqlite3.connect(db_path)) as con, con:
    for user_id in ["U0001", "U0002", "U0003", "U0004", "U0005"]:
        con.execute("UPDATE subscriptions SET plan = 'premium', monthly_price_eur = 10.99, updated_at = ? WHERE user_id = ?",
                    ("2026-09-15 08:00:00", user_id))

In [ ]:
# YOUR CODE HERE

### Exercise 6 (Stretch) — A `get_secret` helper
Write `get_secret(name)` that returns a secret from Colab's Secrets panel when running in Colab
(`from google.colab import userdata; userdata.get(name)`), otherwise from the environment variable `name`.
If it is found nowhere, raise a `KeyError` with a helpful message — **without** printing any secret value.

In [ ]:
def get_secret(name):
    # YOUR CODE HERE
    ...


os.environ["DEMO_TOKEN"] = "not-a-real-token"
print(len(get_secret("DEMO_TOKEN")), "characters")
try:
    get_secret("DOES_NOT_EXIST")
except KeyError as error:
    print(error)